# ⚡ Triple GLM Residual Booster (2-Way Joint TE + Multi-Scale Smoothing)
### Kaggle Playground Series S6E11 (Predicting Credit Card Fraud)

Welcome to the **Triple GLM Residual Booster Pipeline** for **PGS-S6E11**.

### 🔬 The Next-Generation Architecture:
1. **2-Way Joint Digit-Key Target Encoding**: In addition to 1D decimal/digit features, we factorize top 2-way categorical and magnitude intersections (e.g. `merchant_category x cents`, `amount_div10 x balance_div10`, `mc x foreign_tx`, `auth x device`) to directly isolate synthetic fraud clusters.
2. **Dual-Scale Smoothing ($m=10, 25$)**: Captures both sharp, high-sensitivity local fraud spikes and smooth, regularized global trends with strict out-of-fold isolation.
3. **Triple GBDT Family Residual Boosting**: We boost all three premier tabular algorithms directly from the optimal GLM linear log-odds margin ($z_i = \mathbf{w}^T \mathbf{x}_i + b$):
   - **LightGBM** (`init_score = z_i`, leaf-wise shallow trees)
   - **XGBoost** (`base_margin = z_i`, GPU hist lossguide shallow trees)
   - **CatBoost** (`baseline = z_i`, GPU symmetric oblivious trees)
4. **Optimal Simplex Blending & 700k Full Refit**: Out-of-fold rank optimization followed by full-dataset refit for maximum test generalization.


In [ ]:
import os
import gc
import time
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import QuantileTransformer, StandardScaler
from scipy.special import expit, logit
from scipy.stats import rankdata
from scipy.optimize import minimize

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

USE_GPU = shutil.which("nvidia-smi") is not None
print(f"CUDA Available: {USE_GPU} | LightGBM: {lgb.__version__} | XGBoost: {xgb.__version__}")


In [ ]:
class Config:
    SEED = 42
    N_FOLDS = 5
    N_SPLITS = 5
    SMOOTH_SHARP = 10.0
    SMOOTH_BROAD = 25.0
    ES_ROUNDS = 250
    
    # Path resolution
    if os.path.exists("/kaggle/input/playground-series-s6e11"):
        DATA_DIR = Path("/kaggle/input/playground-series-s6e11")
        OUTPUT_DIR = Path(".")
    elif os.path.exists("/kaggle/input/competitions/playground-series-s6e11"):
        DATA_DIR = Path("/kaggle/input/competitions/playground-series-s6e11")
        OUTPUT_DIR = Path(".")
    else:
        DATA_DIR = Path("../../dataset")
        if not (DATA_DIR / "train.csv").exists():
            DATA_DIR = Path("pgs-s6e11/dataset")
        OUTPUT_DIR = Path("../../submission")
        if not OUTPUT_DIR.exists():
            OUTPUT_DIR = Path("pgs-s6e11/submission")

Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data Directory:   {Config.DATA_DIR}")
print(f"Output Directory: {Config.OUTPUT_DIR}")


In [ ]:
train_df = pd.read_csv(Config.DATA_DIR / "train.csv")
test_df = pd.read_csv(Config.DATA_DIR / "test.csv")
sample_sub = pd.read_csv(Config.DATA_DIR / "sample_submission.csv")

TARGET, ID_COL = "is_fraud", "id"
BASE_FEATURES = [c for c in test_df.columns if c != ID_COL]
FLAG_COLS = list(train_df.select_dtypes(include="bool").columns)

for df in (train_df, test_df):
    df[FLAG_COLS] = df[FLAG_COLS].astype("int8")

y = train_df[TARGET].astype(int).values

CAT_COLS = [c for c in BASE_FEATURES if not pd.api.types.is_numeric_dtype(train_df[c])]
NUM_COLS = [c for c in BASE_FEATURES if c not in CAT_COLS + FLAG_COLS]

# Consistent categorical encoding
for c in CAT_COLS:
    train_df[c] = train_df[c].astype(str)
    test_df[c] = test_df[c].astype(str)

print(f"Loaded: {len(train_df):,} Train rows, {len(test_df):,} Test rows")
print(f"Features: {len(NUM_COLS)} numeric, {len(FLAG_COLS)} flags, {len(CAT_COLS)} categoricals")
print(f"Fraud Rate: {y.mean()*100:.3f}% ({y.sum():,} fraud txns)")


In [ ]:
both_num = pd.concat([train_df[NUM_COLS], test_df[NUM_COLS]])
COUNT_COLS = []
for c in NUM_COLS:
    counts = both_num[c].value_counts()
    for df in (train_df, test_df):
        df[f"{c}_count"] = df[c].map(counts).fillna(0).astype(np.float32)
    COUNT_COLS.append(f"{c}_count")
del both_num; gc.collect()

def n_decimals(v, max_d=4):
    v = v[~np.isnan(v)][:200_000]
    for d in range(max_d + 1):
        scaled = v * 10 ** d
        if np.allclose(scaled, np.rint(scaled), rtol=0, atol=1e-6):
            return d
    return max_d

DECIMALS = {c: n_decimals(train_df[c].values.astype(float)) for c in NUM_COLS}
CENT_COLS = []
for c in NUM_COLS:
    if DECIMALS[c] >= 2:
        for df in (train_df, test_df):
            df[f"{c}_cents"] = (np.abs(np.rint(df[c].values * 10 ** DECIMALS[c])).astype(np.int64) % 100).astype(np.float32)
        CENT_COLS.append(f"{c}_cents")

print(f"Extracted {len(COUNT_COLS)} count features and {len(CENT_COLS)} cents features.")


In [ ]:
def digit_keys(x, d):
    n = np.rint(x * 10 ** d).astype(np.int64)
    a = np.abs(n)
    ip = a // 10 ** d
    nd = np.where(ip > 0, np.floor(np.log10(np.maximum(ip, 1))).astype(np.int64) + 1, 0)
    keys = {"value": n}
    if d >= 1:
        keys["last_digit"] = a % 10
        keys["first_decimal"] = (a // 10 ** (d - 1)) % 10
    if d >= 2:
        keys["cents"] = a % 100
        keys["int_10x"] = n // 10 ** (d - 1)
    keys["int"] = n // 10 ** d
    keys["int_div10"] = n // 10 ** (d + 1)
    keys["lead1"] = nd * 10 + ip // 10 ** np.maximum(nd - 1, 0)
    keys["lead2"] = nd * 100 + ip // 10 ** np.maximum(nd - 2, 0)
    return keys

both = pd.concat([train_df, test_df], ignore_index=True)
KEY_CODES, KEY_K = {}, {}

# 1. Individual 1D Digit & Value Keys
for c in NUM_COLS:
    x = np.nan_to_num(both[c].values.astype(float), nan=-1e9)
    kept = []
    for kname, vals in digit_keys(x, DECIMALS[c]).items():
        codes, uniques = pd.factorize(vals)
        if len(uniques) < 2 or any(np.array_equal(codes, KEY_CODES[k]) for k in kept):
            continue
        name = f"{c}__{kname}"
        KEY_CODES[name], KEY_K[name] = codes.astype(np.int32), len(uniques)
        kept.append(name)

for c in CAT_COLS:
    codes, uniques = pd.factorize(both[c].astype(str))
    name = f"{c}__value"
    KEY_CODES[name], KEY_K[name] = codes.astype(np.int32), len(uniques)

# 2. Key 2-Way Joint Interaction Keys (The Multipliers)
cents_str = (np.abs(np.rint(both["amount_usd"].values * 100)).astype(np.int64) % 100).astype(str)
amt_div10_str = (np.abs(np.rint(both["amount_usd"].values)) // 10).astype(np.int64).astype(str)
bal_div10_str = (np.abs(np.rint(both["account_balance_usd"].values)) // 10).astype(np.int64).astype(str)

joint_pairs = {
    "mc_x_cents": both["merchant_category"] + "_" + cents_str,
    "mc_x_foreign": both["merchant_category"] + "_" + both["is_foreign_transaction"].astype(str),
    "mc_x_scam": both["merchant_category"] + "_" + both["is_ai_generated_scam_attempt"].astype(str),
    "amt10_x_bal10": amt_div10_str + "_" + bal_div10_str,
    "auth_x_dev": both["auth_method"] + "_" + both["device_type"],
    "chan_x_dev": both["channel"] + "_" + both["device_type"],
    "ip_x_foreign": both["ip_country_mismatch"].astype(str) + "_" + both["is_foreign_transaction"].astype(str),
    "vpn_x_ip": both["used_vpn"].astype(str) + "_" + both["ip_country_mismatch"].astype(str),
    "hour_x_dow": both["time_of_day_hour"].astype(str) + "_" + both["day_of_week"].astype(str),
    "card_x_chan": both["card_type"] + "_" + both["channel"],
    "mc_x_hour": both["merchant_category"] + "_" + both["time_of_day_hour"].astype(str),
    "cents_x_scam": cents_str + "_" + both["is_ai_generated_scam_attempt"].astype(str),
}

for kname, s in joint_pairs.items():
    codes, uniques = pd.factorize(s)
    KEY_CODES[f"joint__{kname}"], KEY_K[f"joint__{kname}"] = codes.astype(np.int32), len(uniques)

del both; gc.collect()

KEY_NAMES = list(KEY_CODES)
TE_COLS = [f"te_{k}" for k in KEY_NAMES]

# Shared category encoding for GBDTs
for c in CAT_COLS:
    cats = sorted(set(train_df[c]) | set(test_df[c]))
    train_df[c] = pd.Categorical(train_df[c], categories=cats)
    test_df[c] = pd.Categorical(test_df[c], categories=cats)

TREE_FEATS = BASE_FEATURES + CENT_COLS + COUNT_COLS + TE_COLS
RAW_ALL = BASE_FEATURES + CENT_COLS + COUNT_COLS

print(f"Total encoded keys: {len(KEY_NAMES)} (including {len(joint_pairs)} 2-way joint keys)")
print(f"Total features for tree models: {len(TREE_FEATS)}")


In [ ]:
N_TR = len(train_df)
skf = StratifiedKFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=Config.SEED)
folds = np.zeros(N_TR, dtype=int)
for f, (_, va) in enumerate(skf.split(train_df, y)):
    folds[va] = f

GROUPS = np.concatenate([folds, np.full(len(test_df), -1)])

def target_encode(stat_rows, apply_rows, yv, smooth=Config.SMOOTH_SHARP, exclude_own_fold=True):
    ys = yv[stat_rows].astype(float)
    prior = ys.mean()
    G = GROUPS
    g_stat, g_app = G[stat_rows], G[apply_rows]
    fold_ids = np.unique(g_stat) if exclude_own_fold else []
    out = np.empty((len(apply_rows), len(KEY_NAMES)), dtype=np.float32)
    
    for j, k in enumerate(KEY_NAMES):
        codes, K = KEY_CODES[k], KEY_K[k]
        cs, ca = codes[stat_rows], codes[apply_rows]
        s = np.bincount(cs, weights=ys, minlength=K)
        n = np.bincount(cs, minlength=K).astype(float)
        out[:, j] = (s[ca] + prior * smooth) / (n[ca] + smooth)
        
        for g in fold_ids:
            m_app = g_app == g
            if not m_app.any():
                continue
            m_st = g_stat == g
            sg = np.bincount(cs[m_st], weights=ys[m_st], minlength=K)
            ng = np.bincount(cs[m_st], minlength=K)
            cg = ca[m_app]
            out[m_app, j] = (s[cg] - sg[cg] + prior * smooth) / (n[cg] - ng[cg] + smooth)
            
    return out

def enc_frame(df, rows, block):
    part = df[RAW_ALL] if rows is None else df[RAW_ALL].iloc[rows]
    return pd.concat([part.reset_index(drop=True), pd.DataFrame(block, columns=TE_COLS)], axis=1)

def fold_frames(f):
    tr_idx, va_idx = np.where(folds != f)[0], np.where(folds == f)[0]
    te = target_encode(tr_idx, np.concatenate([tr_idx, va_idx, N_TR + np.arange(len(test_df))]), y)
    a, b = len(tr_idx), len(tr_idx) + len(va_idx)
    return (tr_idx, va_idx,
            enc_frame(train_df, tr_idx, te[:a]),
            enc_frame(train_df, va_idx, te[a:b]),
            enc_frame(test_df, None, te[b:]))

print("Target encoding pipeline with 2-way joint keys ready.")


In [ ]:
def glm_matrix(X, qt=None, sc=None):
    p = np.clip(X[TE_COLS].values, 1e-4, 1 - 1e-4)
    num = X[NUM_COLS + COUNT_COLS].values.astype(float)
    if qt is None:
        qt = QuantileTransformer(n_quantiles=1000, output_distribution="normal", subsample=200_000, random_state=Config.SEED).fit(num)
    M = np.hstack([np.log(p / (1 - p)), qt.transform(num), X[FLAG_COLS].values.astype(float)])
    if sc is None:
        sc = StandardScaler().fit(M)
    return np.nan_to_num(sc.transform(M)), qt, sc

def glm_fit(X, yv):
    M, qt, sc = glm_matrix(X)
    m = LogisticRegression(C=0.5, max_iter=500, random_state=Config.SEED).fit(M, yv)
    return lambda Z: m.decision_function(glm_matrix(Z, qt, sc)[0])

# 1. LightGBM Residual Booster
TE_LGB_PARAMS = dict(
    objective="binary", metric="auc", boosting_type="gbdt", learning_rate=0.03,
    num_leaves=4, max_depth=3, min_child_samples=100,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0,
    random_state=Config.SEED, n_jobs=-1, verbose=-1
)

def lgb_fit(X, yv, X_es=None, y_es=None, margin=None, margin_es=None, n=None):
    trn_data = lgb.Dataset(X[TREE_FEATS], label=yv, init_score=margin)
    if X_es is None:
        bst = lgb.train(TE_LGB_PARAMS, trn_data, num_boost_round=n)
        return bst, n
    val_data = lgb.Dataset(X_es[TREE_FEATS], label=y_es, init_score=margin_es, reference=trn_data)
    bst = lgb.train(TE_LGB_PARAMS, trn_data, num_boost_round=10000, valid_sets=[val_data],
                    callbacks=[lgb.early_stopping(Config.ES_ROUNDS, verbose=False)])
    return bst, bst.best_iteration

def lgb_pred(bst, X, n=None, margin=None):
    raw = bst.predict(X[TREE_FEATS], num_iteration=n, raw_score=True)
    return expit(margin + raw) if margin is not None else expit(raw)

# 2. XGBoost Residual Booster
TE_XGB_PARAMS = dict(
    objective="binary:logistic", eval_metric="auc", tree_method="hist",
    device="cuda" if USE_GPU else "cpu", grow_policy="lossguide", max_leaves=4,
    learning_rate=0.03, subsample=0.8, colsample_bytree=0.5, reg_lambda=1.0,
    random_state=Config.SEED
)

def xgb_fit(X, yv, X_es=None, y_es=None, margin=None, margin_es=None, n=None):
    dtrain = xgb.DMatrix(X[TREE_FEATS], label=yv, enable_categorical=True) if margin is None else xgb.DMatrix(X[TREE_FEATS], label=yv, base_margin=margin, enable_categorical=True)
    if X_es is None:
        bst = xgb.train(TE_XGB_PARAMS, dtrain, num_boost_round=n)
        return bst, n
    dval = xgb.DMatrix(X_es[TREE_FEATS], label=y_es, enable_categorical=True) if margin_es is None else xgb.DMatrix(X_es[TREE_FEATS], label=y_es, base_margin=margin_es, enable_categorical=True)
    bst = xgb.train(TE_XGB_PARAMS, dtrain, num_boost_round=10000, evals=[(dval, "val")],
                    early_stopping_rounds=Config.ES_ROUNDS, verbose_eval=False)
    return bst, bst.best_iteration + 1

def xgb_pred(bst, X, n=None, margin=None):
    dx = xgb.DMatrix(X[TREE_FEATS], enable_categorical=True) if margin is None else xgb.DMatrix(X[TREE_FEATS], base_margin=margin, enable_categorical=True)
    return bst.predict(dx, iteration_range=(0, n)) if n is not None else bst.predict(dx)

# 3. CatBoost Residual Booster
def cb_fit(X, yv, X_es=None, y_es=None, margin=None, margin_es=None, n=None):
    cb_params = dict(
        iterations=n or 10000, learning_rate=0.03, depth=4, eval_metric="AUC",
        random_seed=Config.SEED, task_type="GPU" if USE_GPU else "CPU", verbose=False
    )
    if X_es is not None:
        cb_params["early_stopping_rounds"] = Config.ES_ROUNDS
    m = CatBoostClassifier(**cb_params)
    p_tr = Pool(X[TREE_FEATS], yv, cat_features=CAT_COLS, baseline=margin)
    if X_es is None:
        m.fit(p_tr)
        return m, n
    p_es = Pool(X_es[TREE_FEATS], y_es, cat_features=CAT_COLS, baseline=margin_es)
    m.fit(p_tr, eval_set=p_es)
    return m, m.get_best_iteration() + 1

def cb_pred(m, X, margin=None):
    p = Pool(X[TREE_FEATS], cat_features=CAT_COLS, baseline=margin)
    return m.predict_proba(p)[:, 1]


In [ ]:
MODELS = ["glm", "lgb_glm", "xgb_glm", "cb_glm"]
res = {k: {"oof": np.zeros(N_TR), "test": np.zeros(len(test_df)), "folds": []} for k in MODELS}
iters = {k: [] for k in ["lgb_glm", "xgb_glm", "cb_glm"]}

print("=" * 75)
print("Starting 5-Fold Triple GLM Residual Boosting (LGB + XGB + CatBoost)...")
print("=" * 75)

start_time = time.time()

for fold in range(Config.N_FOLDS):
    fold_t0 = time.time()
    tr_idx, va_idx, X_tr, X_va, X_te = fold_frames(fold)
    y_tr, y_va = y[tr_idx], y[va_idx]
    
    fit_i, es_i = train_test_split(np.arange(len(tr_idx)), test_size=0.10, stratify=y_tr, random_state=Config.SEED + fold)
    X_fit, X_es, y_fit, y_es = X_tr.iloc[fit_i], X_tr.iloc[es_i], y_tr[fit_i], y_tr[es_i]
    
    preds = {}
    
    # 1. GLM Convex Hyperplane
    glm = glm_fit(X_fit, y_fit)
    mg = {k: glm(X) for k, X in [("fit", X_fit), ("es", X_es), ("va", X_va), ("te", X_te)]}
    preds["glm"] = (expit(mg["va"]), expit(mg["te"]))
    
    # 2. LightGBM Residual Booster
    m_lgb, n_lgb = lgb_fit(X_fit, y_fit, X_es, y_es, margin=mg["fit"], margin_es=mg["es"])
    iters["lgb_glm"].append(n_lgb)
    preds["lgb_glm"] = (lgb_pred(m_lgb, X_va, n_lgb, mg["va"]), lgb_pred(m_lgb, X_te, n_lgb, mg["te"]))
    
    # 3. XGBoost Residual Booster
    m_xgb, n_xgb = xgb_fit(X_fit, y_fit, X_es, y_es, margin=mg["fit"], margin_es=mg["es"])
    iters["xgb_glm"].append(n_xgb)
    preds["xgb_glm"] = (xgb_pred(m_xgb, X_va, n_xgb, mg["va"]), xgb_pred(m_xgb, X_te, n_xgb, mg["te"]))
    
    # 4. CatBoost Residual Booster
    m_cb, n_cb = cb_fit(X_fit, y_fit, X_es, y_es, margin=mg["fit"], margin_es=mg["es"])
    iters["cb_glm"].append(n_cb)
    preds["cb_glm"] = (cb_pred(m_cb, X_va, mg["va"]), cb_pred(m_cb, X_te, mg["te"]))
    
    for k, (pv, pt) in preds.items():
        res[k]["oof"][va_idx] = pv
        res[k]["test"] += pt / Config.N_FOLDS
        res[k]["folds"].append(roc_auc_score(y_va, pv))
        
    fold_elapsed = time.time() - fold_t0
    print(f"Fold {fold+1}/{Config.N_FOLDS} ({fold_elapsed:.1f}s) | " +
          f"GLM: {res['glm']['folds'][-1]:.5f} | " +
          f"LGB: {res['lgb_glm']['folds'][-1]:.5f} | " +
          f"XGB: {res['xgb_glm']['folds'][-1]:.5f} | " +
          f"CB: {res['cb_glm']['folds'][-1]:.5f}")
    
    del X_tr, X_va, X_te, X_fit, X_es, m_lgb, m_xgb, m_cb; gc.collect()

print("=" * 75)
print("5-Fold Cross Validation Complete!")
print("=" * 75)

for k in MODELS:
    res[k]["cv_auc"] = roc_auc_score(y, res[k]["oof"])
    res[k]["pr_auc"] = average_precision_score(y, res[k]["oof"])
    print(f"{k.upper():<10} 5-Fold OOF ROC-AUC: {res[k]['cv_auc']:.5f} | PR-AUC: {res[k]['pr_auc']:.5f}")


In [ ]:
r_xgb = rankdata(res["xgb_glm"]["oof"]) / N_TR
r_lgb = rankdata(res["lgb_glm"]["oof"]) / N_TR
r_cb = rankdata(res["cb_glm"]["oof"]) / N_TR

# Optimize weights via Nelder-Mead on ROC-AUC
def loss(w):
    w1, w2, w3 = np.exp(w) / np.sum(np.exp(w))
    blend = w1 * r_xgb + w2 * r_lgb + w3 * r_cb
    return -roc_auc_score(y, blend)

opt = minimize(loss, [0.0, 0.0, 0.0], method="Nelder-Mead")
w_opt = np.exp(opt.x) / np.sum(np.exp(opt.x))
w_xgb, w_lgb, w_cb = w_opt

oof_blend = w_xgb * r_xgb + w_lgb * r_lgb + w_cb * r_cb
t_xgb = rankdata(res["xgb_glm"]["test"]) / len(test_df)
t_lgb = rankdata(res["lgb_glm"]["test"]) / len(test_df)
t_cb = rankdata(res["cb_glm"]["test"]) / len(test_df)
test_blend = w_xgb * t_xgb + w_lgb * t_lgb + w_cb * t_cb

res["blend"] = {
    "oof": oof_blend,
    "test": test_blend,
    "cv_auc": roc_auc_score(y, oof_blend),
    "pr_auc": average_precision_score(y, oof_blend),
    "weights": {"xgb": w_xgb, "lgb": w_lgb, "cb": w_cb}
}

print(f"Optimal Triple Blend: {w_xgb*100:.1f}% XGB + {w_lgb*100:.1f}% LGB + {w_cb*100:.1f}% CatBoost")
print(f"Ensemble 5-Fold OOF ROC-AUC: {res['blend']['cv_auc']:.5f} | PR-AUC: {res['blend']['pr_auc']:.5f}")


In [ ]:
print("=" * 75)
print("Fitting full-data refit on all 700k rows (1.25x iterations)...")
print("=" * 75)

refit_t0 = time.time()
all_tr = np.arange(N_TR)
te_all = target_encode(all_tr, np.concatenate([all_tr, N_TR + np.arange(len(test_df))]), y)
X_all = enc_frame(train_df, None, te_all[:N_TR])
X_test_all = enc_frame(test_df, None, te_all[N_TR:])
del te_all; gc.collect()

# Refit GLM
glm_full = glm_fit(X_all, y)
mg_all, mg_test = glm_full(X_all), glm_full(X_test_all)
res["glm"]["refit"] = expit(mg_test)

# Refit LGB-GLM
n_lgb_refit = int(np.mean(iters["lgb_glm"]) * 1.25)
m_lgb_full, _ = lgb_fit(X_all, y, margin=mg_all, n=n_lgb_refit)
res["lgb_glm"]["refit"] = lgb_pred(m_lgb_full, X_test_all, n_lgb_refit, mg_test)

# Refit XGB-GLM
n_xgb_refit = int(np.mean(iters["xgb_glm"]) * 1.25)
m_xgb_full, _ = xgb_fit(X_all, y, margin=mg_all, n=n_xgb_refit)
res["xgb_glm"]["refit"] = xgb_pred(m_xgb_full, X_test_all, n_xgb_refit, mg_test)

# Refit CatBoost-GLM
n_cb_refit = int(np.mean(iters["cb_glm"]) * 1.25)
m_cb_full, _ = cb_fit(X_all, y, margin=mg_all, n=n_cb_refit)
res["cb_glm"]["refit"] = cb_pred(m_cb_full, X_test_all, mg_test)

# Refit Blend
t_refit_xgb = rankdata(res["xgb_glm"]["refit"]) / len(test_df)
t_refit_lgb = rankdata(res["lgb_glm"]["refit"]) / len(test_df)
t_refit_cb = rankdata(res["cb_glm"]["refit"]) / len(test_df)
res["blend"]["refit"] = w_xgb * t_refit_xgb + w_lgb * t_refit_lgb + w_cb * t_refit_cb

del X_all, X_test_all, m_lgb_full, m_xgb_full, m_cb_full; gc.collect()
print(f"Full refit finished in {(time.time() - refit_t0)/60:.1f} minutes.")


In [ ]:
# Save Out-Of-Fold predictions for downstream master stacking
np.save(Config.OUTPUT_DIR / "oof_triple_glm.npy", res["glm"]["oof"])
np.save(Config.OUTPUT_DIR / "oof_triple_lgb.npy", res["lgb_glm"]["oof"])
np.save(Config.OUTPUT_DIR / "oof_triple_xgb.npy", res["xgb_glm"]["oof"])
np.save(Config.OUTPUT_DIR / "oof_triple_cb.npy", res["cb_glm"]["oof"])
np.save(Config.OUTPUT_DIR / "oof_triple_blend.npy", res["blend"]["oof"])

# Generate Submissions
sub_blend = sample_sub.copy()
sub_blend["is_fraud"] = res["blend"]["refit"]
sub_blend.to_csv(Config.OUTPUT_DIR / "triple_glm_joint_te_blend_submission.csv", index=False)

sub_xgb = sample_sub.copy()
sub_xgb["is_fraud"] = res["xgb_glm"]["refit"]
sub_xgb.to_csv(Config.OUTPUT_DIR / "triple_glm_joint_te_xgb_submission.csv", index=False)

sub_lgb = sample_sub.copy()
sub_lgb["is_fraud"] = res["lgb_glm"]["refit"]
sub_lgb.to_csv(Config.OUTPUT_DIR / "triple_glm_joint_te_lgb_submission.csv", index=False)

sub_cb = sample_sub.copy()
sub_cb["is_fraud"] = res["cb_glm"]["refit"]
sub_cb.to_csv(Config.OUTPUT_DIR / "triple_glm_joint_te_cb_submission.csv", index=False)

print(f"Saved: triple_glm_joint_te_blend_submission.csv ({len(sub_blend):,} rows)")
print(f"Saved: OOF arrays saved to {Config.OUTPUT_DIR}")


In [ ]:
summary_df = pd.DataFrame({
    "Model": ["GLM (Logistic Regression)", "LightGBM GLM-Boost", "XGBoost GLM-Boost", "CatBoost GLM-Boost", "Optimal Triple Simplex Blend"],
    "5-Fold OOF ROC-AUC": [res["glm"]["cv_auc"], res["lgb_glm"]["cv_auc"], res["xgb_glm"]["cv_auc"], res["cb_glm"]["cv_auc"], res["blend"]["cv_auc"]],
    "5-Fold OOF PR-AUC": [res["glm"]["pr_auc"], res["lgb_glm"]["pr_auc"], res["xgb_glm"]["pr_auc"], res["cb_glm"]["pr_auc"], res["blend"]["pr_auc"]]
})

print("-" * 75)
print("TRIPLE GLM RESIDUAL BOOSTING BENCHMARK RESULTS")
print("-" * 75)
print(summary_df.to_string(index=False))

# Plot ROC & PR Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for m in ["glm", "lgb_glm", "xgb_glm", "cb_glm", "blend"]:
    pred = res[m]["oof"]
    label = f"{m.upper()} (AUC = {res[m]['cv_auc']:.5f})"
    from sklearn.metrics import roc_curve, precision_recall_curve
    fpr, tpr, _ = roc_curve(y, pred)
    ax1.plot(fpr, tpr, label=label, lw=2)
    
    prec, rec, _ = precision_recall_curve(y, pred)
    ax2.plot(rec, prec, label=f"{m.upper()} (PR = {res[m]['pr_auc']:.5f})", lw=2)

ax1.plot([0, 1], [0, 1], "k--", lw=1)
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.set_title("ROC Curves (5-Fold Out-Of-Fold)")
ax1.legend(loc="lower right")

ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.set_title("Precision-Recall Curves (5-Fold Out-Of-Fold)")
ax2.legend(loc="lower left")

plt.tight_layout()
plt.savefig(Config.OUTPUT_DIR / "triple_glm_evaluation_curves.png", dpi=150)
plt.show()
